# ⚙️ 02 – Optimisation avec SciPy

---

## 🎯 Objectifs
- Comprendre ce qu'est **optimiser** sans formule mathématique
- Trouver automatiquement le **meilleur paramètre** d'une fonction avec `minimize()`
- Appliquer l'optimisation à des cas **concrets métier** : prix, budget, stock
- Comprendre les **contraintes** et les **bornes** dans une optimisation

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
print("SciPy Optimize prêt ✅")

---
## 📝 Partie 1 – Qu'est-ce qu'optimiser ?

### 🔎 L'idée en langage humain

**Optimiser**, c'est trouver la valeur d'un paramètre qui donne le **meilleur résultat possible**.

**Exemples du quotidien :**
- Une pizzeria veut fixer le **prix** qui maximise son bénéfice
- Un e-commerce veut répartir son **budget pub** entre Google et Facebook pour maximiser les ventes
- Un entrepôt veut trouver la **quantité à commander** qui minimise les coûts de stockage

Dans tous ces cas, on a :
- **Une fonction** qui calcule le résultat selon le paramètre (bénéfice, ventes, coût)
- **Un paramètre** à ajuster (prix, budget, quantité)
- **Un objectif** : trouver le paramètre qui maximise ou minimise le résultat

### 🔎 Comment SciPy résout ce problème ?

SciPy utilise `minimize()` — une fonction qui teste intelligemment des valeurs et converge vers la meilleure.

```
Étape 1 : on part d'un point de départ (x0)
Étape 2 : SciPy regarde la "pente" de la fonction autour de ce point
Étape 3 : SciPy se déplace dans la direction qui améliore le résultat
Étape 4 : répète jusqu'à trouver le point où ça ne s'améliore plus
```

**Attention :** `minimize()` cherche toujours un **minimum** (le point le plus bas).  
Pour trouver un **maximum**, on minimise la fonction **négative** : `minimize(-f(x))` au lieu de `maximize(f(x))`.

```
Maximiser f(x)  =  Minimiser -f(x)

      f(x)                  -f(x)
        ▲                      ▼
    ────┼────              ────┼────
   /   MAX  \              \  MIN  /
  /    ↑    \              /  ↓   \
 /     │     \            /   │    \
                           ← même point, logique inversée
```

---
## 📝 Partie 2 – Premier exemple : le prix optimal d'une pizza

### 🔎 Le problème

Une pizzeria veut fixer le prix de sa pizza Margherita. Elle sait par expérience :
- Si le prix est **trop bas** → elle vend beaucoup mais gagne peu par pizza
- Si le prix est **trop haut** → les clients vont ailleurs, les ventes chutent
- Il existe un **prix idéal** quelque part entre les deux

On modélise le bénéfice journalier selon le prix `p` :

```
benefice(p) = -(p - 8)² + 64

Cette formule dit :
- Le bénéfice maximum est 64€ quand p = 8€
- Plus on s'éloigne de 8€ dans un sens ou dans l'autre, le bénéfice diminue
```

### 🔎 Syntaxe de `minimize()`

```python
from scipy.optimize import minimize

result = minimize(
    fun = fonction_a_minimiser,  # la fonction
    x0  = valeur_de_depart       # point de départ (estimation initiale)
)

result.x      # → le paramètre optimal trouvé (tableau)
result.x[0]   # → la valeur scalaire
result.fun    # → la valeur de la fonction au point optimal
result.success # → True si l'optimisation a convergé
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Fonction de bénéfice : maximum à p=8
def benefice(p):
    return -(p - 8)**2 + 64

# Visualiser la courbe de bénéfice
prix = np.linspace(0, 16, 200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prix, benefice(prix), color="steelblue", linewidth=2.5)
ax.set_title("Bénéfice journalier selon le prix")
ax.set_xlabel("Prix de la pizza (€)")
ax.set_ylabel("Bénéfice (€)")
ax.axhline(0, color="gray", linestyle="-", alpha=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Bénéfice à 5€ :", benefice(5), "€")
print("Bénéfice à 8€ :", benefice(8), "€  ← maximum")
print("Bénéfice à 12€ :", benefice(12), "€")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def benefice(p):
    return -(p - 8)**2 + 64

# minimize() cherche un MINIMUM → on lui donne -benefice pour trouver le MAXIMUM
result = minimize(
    fun = lambda p: -benefice(p),   # on minimise le négatif = on maximise
    x0  = 5                          # on part de 5€ comme estimation initiale
)

prix_optimal  = result.x[0]          # le prix trouvé
benef_optimal = benefice(prix_optimal)  # le bénéfice à ce prix

print(f"Prix optimal    : {prix_optimal:.2f} €")
print(f"Bénéfice maximal: {benef_optimal:.2f} €")
print(f"Convergé        : {result.success}")

# Visualiser le résultat
prix = np.linspace(0, 16, 200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prix, benefice(prix), color="steelblue", linewidth=2.5, label="Bénéfice")
ax.scatter([prix_optimal], [benef_optimal], color="red", s=150, zorder=5,
           label=f"Prix optimal : {prix_optimal:.1f}€ → {benef_optimal:.0f}€")
ax.annotate(
    f"Prix optimal\n{prix_optimal:.1f}€",
    xy=(prix_optimal, benef_optimal),
    xytext=(prix_optimal + 2, benef_optimal - 15),
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red", fontsize=10
)
ax.set_title("Prix optimal trouvé par SciPy")
ax.set_xlabel("Prix (€)")
ax.set_ylabel("Bénéfice (€)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📝 Partie 3 – Optimisation avec bornes

### 🔎 Pourquoi des bornes ?

Dans la vraie vie, les paramètres ont des **contraintes physiques ou commerciales** :
- Un prix ne peut pas être négatif
- Un budget ne peut pas dépasser un plafond
- Une quantité commandée doit être entre 10 et 500 unités

On passe les bornes avec le paramètre `bounds` :

```python
result = minimize(
    fun    = ma_fonction,
    x0     = valeur_initiale,
    bounds = [(min, max)]    # liste de tuples : une paire (min, max) par paramètre
)

# Exemples :
bounds = [(0, None)]      # prix ≥ 0, pas de maximum
bounds = [(5, 15)]        # prix entre 5 et 15
bounds = [(0, 1000)]      # quantité entre 0 et 1000
```

### 🔎 Optimisation multi-paramètres

`minimize()` peut aussi optimiser **plusieurs paramètres en même temps**. Dans ce cas, la fonction reçoit un tableau `x` et `x[0]`, `x[1]`… sont les différents paramètres.

```python
def cout(x):
    budget_google   = x[0]
    budget_facebook = x[1]
    return -(ventes_google(budget_google) + ventes_facebook(budget_facebook))

result = minimize(
    fun    = cout,
    x0     = [500, 500],         # estimation initiale : 500€ sur chaque
    bounds = [(0, 1000), (0, 1000)]  # bornes pour chaque paramètre
)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Situation : un restaurant peut fixer son menu entre 8€ et 20€
# La demande diminue quand le prix monte, le coût est fixe à 4€/repas

def profit_restaurant(prix):
    """Prix en euros, retourne le profit journalier"""
    cout_unitaire = 4.0
    # Demande : plus le prix est haut, moins de clients
    nb_clients = max(0, 200 - 10 * prix)   # 200 clients à 0€, 0 client à 20€
    marge = prix - cout_unitaire
    return -(nb_clients * marge)            # négatif car on veut maximiser

# Sans bornes
result_libre = minimize(profit_restaurant, x0=10)
print(f"Sans bornes — Prix optimal : {result_libre.x[0]:.2f} €")
print(f"             Profit        : {-result_libre.fun:.2f} €/jour")

# Avec bornes : le prix doit être entre 8€ et 20€ (contrainte commerciale)
result_borne = minimize(profit_restaurant, x0=10, bounds=[(8, 20)])
print(f"\nAvec bornes [8-20€] — Prix optimal : {result_borne.x[0]:.2f} €")
print(f"                      Profit        : {-result_borne.fun:.2f} €/jour")

# Visualisation
prix_range = np.linspace(4, 22, 200)
profits    = [-profit_restaurant(p) for p in prix_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prix_range, profits, color="steelblue", linewidth=2)
ax.axvspan(8, 20, color="green", alpha=0.1, label="Zone autorisée [8-20€]")
ax.scatter([result_libre.x[0]], [-result_libre.fun], color="red",
           s=120, zorder=5, label=f"Optimal libre : {result_libre.x[0]:.1f}€")
ax.scatter([result_borne.x[0]], [-result_borne.fun], color="orange",
           s=120, zorder=5, marker="s", label=f"Optimal borné : {result_borne.x[0]:.1f}€")
ax.set_title("Profit journalier — avec et sans contrainte de prix")
ax.set_xlabel("Prix du menu (€)")
ax.set_ylabel("Profit (€/jour)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Situation : répartir un budget pub de 1000€ entre Google et Facebook
# pour maximiser les ventes
# Google : rendements décroissants — moins efficace en grande quantité
# Facebook : plus efficace sur les petits budgets

def ventes_google(budget):
    return 50 * np.sqrt(budget)     # racine carrée = rendements décroissants

def ventes_facebook(budget):
    return 0.08 * budget + 30       # linéaire, bonne efficacité de base

def objectif(x):
    bg = x[0]   # budget Google
    bf = x[1]   # budget Facebook
    return -(ventes_google(bg) + ventes_facebook(bf))   # négatif → on maximise

budget_total = 1000

result = minimize(
    fun    = objectif,
    x0     = [500, 500],              # départ : 500€ sur chaque
    bounds = [(0, budget_total), (0, budget_total)],
    # Contrainte : budget_google + budget_facebook = 1000
    constraints = [{"type": "eq", "fun": lambda x: x[0] + x[1] - budget_total}]
)

bg_opt = result.x[0]
bf_opt = result.x[1]
ventes_max = -result.fun

print("=== Répartition optimale du budget pub ===")
print(f"Budget Google   : {bg_opt:.0f} € → {ventes_google(bg_opt):.0f} ventes")
print(f"Budget Facebook : {bf_opt:.0f} € → {ventes_facebook(bf_opt):.0f} ventes")
print(f"Total budget    : {bg_opt + bf_opt:.0f} €")
print(f"Ventes totales  : {ventes_max:.0f}")
print()
print("Comparaison — répartition 50/50 :")
v_5050 = ventes_google(500) + ventes_facebook(500)
print(f"  50/50 : {v_5050:.0f} ventes")
print(f"  Gain optimal vs 50/50 : +{ventes_max - v_5050:.0f} ventes")

---
## 🧩 Exercice 1 – Trouver le prix optimal d'un abonnement

Une plateforme de streaming veut fixer le prix mensuel de son abonnement.

**Modèle :**
- Nombre d'abonnés : `abonnes(p) = 5000 - 200 * p` (diminue avec le prix)
- Coût mensuel par abonné : `2 €`
- Revenu mensuel : `p * abonnes(p)`
- **Profit mensuel** = Revenu - Coûts = `(p - 2) * abonnes(p)`

1. Définissez la fonction `profit(p)` qui calcule le profit mensuel
2. Tracez la courbe de profit pour des prix entre 0 et 25 €
3. Utilisez `minimize()` pour trouver le prix optimal (attention : maximiser !)
4. Ajoutez une contrainte : le prix doit être **entre 3 et 20 €** (bounds)
5. Affichez le nombre d'abonnés et le profit au prix optimal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# TODO 1 : définir les fonctions
def abonnes(p):
    return max(0, 5000 - 200 * p)

def profit(p):
    return ...   # (p - 2) * abonnes(p)

# TODO 2 : tracer la courbe
prix_range = np.linspace(0, 25, 300)
profits    = [profit(p) for p in prix_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prix_range, profits, color=..., linewidth=2)
ax.axhline(0, color="gray", alpha=0.5)
ax.set_title("Profit mensuel selon le prix de l'abonnement")
ax.set_xlabel("Prix mensuel (€)")
ax.set_ylabel("Profit (€)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO 3 : minimiser le négatif du profit
result = minimize(
    fun    = lambda p: -profit(p),
    x0     = ...,
    bounds = [...]
)

p_opt = result.x[0]

# TODO 5 : afficher les résultats
print(f"Prix optimal    : {p_opt:.2f} €/mois")
print(f"Abonnés         : {abonnes(p_opt):.0f}")
print(f"Profit mensuel  : {profit(p_opt):,.0f} €")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def abonnes(p):
    return max(0, 5000 - 200 * p)

def profit(p):
    return (p - 2) * abonnes(p)   # marge × nombre d'abonnés

prix_range = np.linspace(0, 25, 300)
profits    = [profit(p) for p in prix_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prix_range, profits, color="steelblue", linewidth=2)
ax.axhline(0, color="gray", alpha=0.5)
ax.set_title("Profit mensuel selon le prix de l'abonnement")
ax.set_xlabel("Prix mensuel (€)")
ax.set_ylabel("Profit (€)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

result = minimize(
    fun    = lambda p: -profit(p),   # on minimise le négatif = on maximise
    x0     = 10,                      # estimation initiale : 10€
    bounds = [(3, 20)]                # prix entre 3 et 20€
)

p_opt = result.x[0]
print(f"Prix optimal    : {p_opt:.2f} €/mois")
print(f"Abonnés         : {abonnes(p_opt):.0f}")
print(f"Profit mensuel  : {profit(p_opt):,.0f} €")
# → Prix optimal ≈ 13.50 €
# → 2300 abonnés environ
# → Profit ≈ 26 450 €/mois
```
</details>

---
## 🧩 Exercice 2 – Optimiser la quantité à commander

Un responsable logistique veut trouver la **quantité optimale à commander** à chaque réapprovisionnement.

**Le problème de stock :**
- Commander **trop peu** → rupture de stock, clients mécontents, coût de réapprovisionnement fréquent
- Commander **trop** → coûts de stockage élevés, capital immobilisé

**Modèle de coût total annuel :**
```
coût_total(q) = cout_commande(q) + cout_stockage(q)

cout_commande(q) = (demande_annuelle / q) * cout_par_commande
                 = (1200 / q) * 50
                   ↑ plus q est grand, moins on commande souvent

cout_stockage(q) = (q / 2) * cout_par_unite_stockee
                 = (q / 2) * 3
                   ↑ plus q est grand, plus on stocke en moyenne
```

1. Définissez `cout_total(q)` selon la formule ci-dessus
2. Tracez la courbe du coût total pour des quantités entre 1 et 500
3. Tracez aussi les deux composantes séparément (commande et stockage)
4. Trouvez la quantité optimale avec `minimize()` (bornes : entre 1 et 500)
5. Affichez le coût total minimal et le nombre de commandes par an

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

demande_annuelle   = 1200   # unités/an
cout_par_commande  = 50     # € par commande passée
cout_stockage_unit = 3      # € par unité stockée par an

# TODO 1 : définir les fonctions de coût
def cout_commande(q):
    return (demande_annuelle / q) * cout_par_commande

def cout_stockage(q):
    return (q / 2) * cout_stockage_unit

def cout_total(q):
    return ...   # cout_commande(q) + cout_stockage(q)

# TODO 2 & 3 : tracer les courbes
quantites = np.linspace(1, 500, 500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(quantites, [cout_total(q)    for q in quantites], color="navy",      linewidth=2.5, label="Coût total")
ax.plot(quantites, [cout_commande(q) for q in quantites], color="coral",     linestyle="--", label="Coût commande")
ax.plot(quantites, [cout_stockage(q) for q in quantites], color="seagreen",  linestyle="--", label="Coût stockage")
ax.set_title("Coût total selon la quantité commandée")
ax.set_xlabel("Quantité commandée (q)")
ax.set_ylabel("Coût annuel (€)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 3000)
plt.tight_layout()
plt.show()

# TODO 4 : minimiser le coût total
result = minimize(
    fun    = cout_total,
    x0     = ...,
    bounds = [...]
)

q_opt = result.x[0]

# TODO 5 : résultats
print(f"Quantité optimale à commander : {q_opt:.0f} unités")
print(f"Coût total minimal            : {cout_total(q_opt):.2f} €/an")
print(f"Nombre de commandes par an    : {demande_annuelle / q_opt:.1f}")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

demande_annuelle   = 1200
cout_par_commande  = 50
cout_stockage_unit = 3

def cout_commande(q):
    return (demande_annuelle / q) * cout_par_commande

def cout_stockage(q):
    return (q / 2) * cout_stockage_unit

def cout_total(q):
    return cout_commande(q) + cout_stockage(q)

quantites = np.linspace(1, 500, 500)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(quantites, [cout_total(q)    for q in quantites], color="navy",     linewidth=2.5, label="Coût total")
ax.plot(quantites, [cout_commande(q) for q in quantites], color="coral",    linestyle="--", label="Coût commande")
ax.plot(quantites, [cout_stockage(q) for q in quantites], color="seagreen", linestyle="--", label="Coût stockage")
ax.set_title("Coût total selon la quantité commandée")
ax.set_xlabel("Quantité commandée"); ax.set_ylabel("Coût annuel (€)")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 3000)
plt.tight_layout(); plt.show()

result = minimize(cout_total, x0=100, bounds=[(1, 500)])
q_opt  = result.x[0]

print(f"Quantité optimale : {q_opt:.0f} unités")
print(f"Coût minimal      : {cout_total(q_opt):.2f} €/an")
print(f"Commandes/an      : {demande_annuelle / q_opt:.1f}")
# → q_opt ≈ 200 unités (formule EOQ = √(2 × D × K / h) = √(2×1200×50/3) = 200)
# → Coût minimal ≈ 600 €/an
# → 6 commandes par an
```

> 💡 **Note :** ce problème a une solution analytique connue (formule EOQ). SciPy retrouve exactement le même résultat, mais sans avoir besoin de connaître la formule !
</details>

---
## ✅ Récapitulatif

| Concept | Code | Ce qu'il faut retenir |
|---------|------|-----------------------|
| **`minimize(f, x0)`** | `from scipy.optimize import minimize` | Trouve le minimum de `f` en partant de `x0` |
| **Maximiser** | `minimize(lambda x: -f(x), x0)` | On minimise le négatif |
| **`result.x[0]`** | — | La valeur optimale trouvée |
| **`result.fun`** | — | La valeur de la fonction au point optimal |
| **Bornes** | `bounds=[(min, max)]` | Limiter le paramètre à un intervalle |
| **Multi-paramètres** | `x0=[val1, val2]`, `bounds=[(m1,M1),(m2,M2)]` | Optimiser plusieurs paramètres en même temps |
| **Contrainte** | `constraints=[{"type":"eq", "fun": lambda x: ...}]` | Imposer une égalité entre paramètres (ex: budget fixe) |

**La règle d'or :** `minimize()` cherche toujours un minimum. Pour maximiser, retournez la fonction négative.

**Cas d'usage en entreprise :**
- **Prix optimal** : maximiser le profit selon le prix
- **Répartition budgétaire** : maximiser les ventes sous contrainte de budget total
- **Gestion de stock** : minimiser les coûts de commande + stockage
- **Planification** : minimiser les coûts de production sous contraintes de capacité

---
📘 **Prochain notebook → `03` : Intégration avec SciPy**